In [1]:
!pip install openmeteo-requests retry-requests requests-cache

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.1/167.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.2/683.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.8/145.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.2 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.2.10
    Uninstalling flatbuffers-25.2.10:
      Successfully uninstalled flatbuffers-25.2.10
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.

In [2]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import matplotlib.pyplot as plt
import os
from datetime import datetime, timedelta

def setup_client():
    """
    Menyiapkan client API dengan fitur Cache dan Retry.
    Tujuannya agar koneksi stabil dan hemat kuota.
    """
    # Buat cache bernama '.cache', data disimpan selama 3600 detik (1 jam)
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)

    # Jika gagal connect, coba lagi (retry) sampai 5x
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)

    # Return objek client yang siap pakai
    return openmeteo_requests.Client(session=retry_session)

In [3]:
# ==========================================
# 2. FETCH DATA PER CHUNK
# ==========================================
def fetch_chunk(client, lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "rain", 
                   "wind_speed_10m", "wind_direction_10m", "surface_pressure", "weather_code"],
        "timezone": "Asia/Jakarta"
    }

    print(f"   ⏳ Mengambil: {start_date} s.d {end_date}...")
    
    # --- LOGIKA RETRY / ANTI-LIMIT ---
    max_retries = 5
    attempt = 0
    
    while attempt < max_retries:
        try:
            # Coba panggil API
            responses = client.weather_api(url, params=params)
            
            # Jika berhasil, langsung proses dan kembalikan data
            return process_data(responses[0]) 

        except Exception as e:
            error_msg = str(e)
            
            # CEK APAKAH ERROR KARENA LIMIT?
            if "Minutely API request limit exceeded" in error_msg or "429" in error_msg:
                print(f"      ⛔ Kena Limit API! Menunggu 70 detik sebelum coba lagi... (Percobaan {attempt+1}/{max_retries})")
                # Tunggu 60 detik + 10 detik buffer biar aman
                time.sleep(70)
                attempt += 1
            else:
                # Jika error lain (misal koneksi putus total), print dan coba lagi sebentar
                print(f"      ❌ Error lain: {e}. Menunggu 5 detik...")
                time.sleep(5)
                attempt += 1
    
    # Jika sudah mencoba 5x masih gagal juga
    print(f"   💀 Gagal total mengambil chunk {start_date} setelah {max_retries} kali percobaan.")
    return None

# ==========================================
# 3. PROCESS DATA (UBAHAN: Date Jadi Kolom Biasa)
# ==========================================
def process_data(response):
    hourly = response.Hourly()

    date_range = pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq = pd.Timedelta(seconds=hourly.Interval()),
        inclusive = "left"
    )

    df = pd.DataFrame(data = {
        "date": date_range, # Date tetap jadi kolom biasa
        "temperature": hourly.Variables(0).ValuesAsNumpy(),
        "humidity": hourly.Variables(1).ValuesAsNumpy(),
        "dewpoint":hourly.Variables(2).ValuesAsNumpy(),
        "rain_mm": hourly.Variables(3).ValuesAsNumpy(),
        "wind_speed": hourly.Variables(4).ValuesAsNumpy(),
        "wind_direction": hourly.Variables(5).ValuesAsNumpy(),
        "pressure": hourly.Variables(6).ValuesAsNumpy(),
        "weather_code": hourly.Variables(7).ValuesAsNumpy()
    })

    # Konversi timezone kolom date (Tanpa set_index)
    df['date'] = df['date'].dt.tz_convert('Asia/Jakarta')
    
    return df

# ==========================================
# 4. LOGIKA UTAMA: CHUNKING & MERGING (SAFE MODE)
# ==========================================
def fetch_long_period_data(client, lat, lon, start_str, end_str, folder_tujuan, chunk_years=10):
    
    # ... (Bagian awal sama) ...
    if not os.path.exists(folder_tujuan):
        os.makedirs(folder_tujuan)

    start_date = datetime.strptime(start_str, "%Y-%m-%d")
    final_end_date = datetime.strptime(end_str, "%Y-%m-%d")
    all_files = []
    current_start = start_date

    print(f"🚀 Memulai Misi Pengambilan Data...")

    # --- FASE 1: DOWNLOAD ---
    while current_start < final_end_date:
        current_end = current_start.replace(year=current_start.year + chunk_years) - timedelta(days=1)
        if current_end > final_end_date:
            current_end = final_end_date

        s_str = current_start.strftime("%Y-%m-%d")
        e_str = current_end.strftime("%Y-%m-%d")
        chunk_filename = os.path.join(folder_tujuan, f"temp_{s_str}_{e_str}.csv")

        if os.path.exists(chunk_filename):
            print(f"⏩ Skip: {s_str} - {e_str} (Sudah ada)")
            all_files.append(chunk_filename)
        else:
            # Panggil fetch_chunk yang SUDAH DIMODIFIKASI di atas
            df_chunk = fetch_chunk(client, lat, lon, s_str, e_str)
            
            if df_chunk is not None:
                df_chunk.to_csv(chunk_filename, index=False)
                print(f"   ✅ Tersimpan: {chunk_filename}")
                all_files.append(chunk_filename)
                
                # Istirahat normal antar chunk (bukan saat error)
                time.sleep(2) 
            else:
                print("   ⚠️ Chunk ini dilewati karena gagal total.")

        current_start = current_end + timedelta(days=1)

    print("\n🔗 Menggabungkan semua pecahan data...")

    # --- FASE 2: MERGE (Sama seperti kode sukses sebelumnya) ---
    df_list = []
    for f in all_files:
        try:
            df = pd.read_csv(f, parse_dates=['date'])
            df_list.append(df)
        except Exception as e:
            print(f"Warning: Gagal membaca file {f}, dilewati.")

    if df_list:
        df_final = pd.concat(df_list, ignore_index=True)
        df_final = df_final.drop_duplicates(subset=['date'], keep='first')
        df_final = df_final.set_index('date')
        df_final = df_final.sort_index()

        print(f"🎉 SUKSES! Total Data: {len(df_final)} baris.")
        return df_final
    else:
        return None

In [4]:
import time
from datetime import date, timedelta
import os
# Pastikan library lain yang dibutuhkan (openmeteo_requests, pandas, retry, dll) sudah terimport di bagian atas filemu

# ==========================================
# FUNGSI TAMBAHAN: RETRY LOGIC (ANTI GAGAL)
# ==========================================
def fetch_chunk_safe(func_fetch, *args, max_retries=5, **kwargs):
    """
    Mencoba menjalankan fungsi fetch. Jika gagal (kena limit API),
    akan menunggu dan mencoba lagi secara otomatis.
    """
    wait_seconds = 60 # Tunggu 1 menit jika gagal (sesuai aturan Open Meteo)

    for attempt in range(1, max_retries + 1):
        try:
            return func_fetch(*args, **kwargs)
        except Exception as e:
            print(f"⚠️ Percobaan ke-{attempt} gagal: {e}")
            if attempt < max_retries:
                print(f"⏳ Menunggu {wait_seconds} detik sebelum mencoba lagi...")
                time.sleep(wait_seconds)
            else:
                print("❌ Gagal total setelah beberapa kali percobaan.")
                raise e # Lempar error jika sudah menyerah

# ==========================================
# EKSEKUSI UTAMA
# ==========================================
if __name__ == "__main__":
    # Konfigurasi
    LAT = -7.736436737566032
    LON = 109.6460550796716
    MULAI = "1950-01-01"
    
    # Set Akhir = Kemarin
    kemarin = date.today() - timedelta(days=1)
    AKHIR = kemarin.strftime("%Y-%m-%d")

    FOLDER = "open_meteo_climate"
    FILE_FINAL = "kebumen_75tahun_lengkap.csv"

    client = setup_client()

    # Jalankan
    try:
        # PENTING: Gunakan chunk_years lebih kecil (misal 5) agar tidak terlalu berat
        # dan mengurangi risiko timeout di sisi server
        df_lengkap = fetch_long_period_data(client, LAT, LON, MULAI, AKHIR, FOLDER, chunk_years=5)

        if df_lengkap is not None:
            path_final = os.path.join(FOLDER, FILE_FINAL)
            # Simpan final
            df_lengkap.to_csv(path_final) 
            print(f"💾 File Gabungan Tersimpan: {path_final}")
            
    except Exception as e:
        print(f"❌ Terjadi Error Fatal: {e}")

🚀 Memulai Misi Pengambilan Data...
   ⏳ Mengambil: 1950-01-01 s.d 1954-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1950-01-01_1954-12-31.csv
   ⏳ Mengambil: 1955-01-01 s.d 1959-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1955-01-01_1959-12-31.csv
   ⏳ Mengambil: 1960-01-01 s.d 1964-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1960-01-01_1964-12-31.csv
   ⏳ Mengambil: 1965-01-01 s.d 1969-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1965-01-01_1969-12-31.csv
   ⏳ Mengambil: 1970-01-01 s.d 1974-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1970-01-01_1974-12-31.csv
   ⏳ Mengambil: 1975-01-01 s.d 1979-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1975-01-01_1979-12-31.csv
   ⏳ Mengambil: 1980-01-01 s.d 1984-12-31...
      ⛔ Kena Limit API! Menunggu 70 detik sebelum coba lagi... (Percobaan 1/5)
   ✅ Tersimpan: open_meteo_climate/temp_1980-01-01_1984-12-31.csv
   ⏳ Mengambil: 1985-01-01 s.d 1989-12-31...
   ✅ Tersimpan: open_meteo_climate/temp_1985-01-01_1989-12-31.cs

In [5]:
data = pd.read_csv("/kaggle/working/open_meteo_climate/kebumen_75tahun_lengkap.csv")
data.tail(10)

,date,temperature,humidity,dewpoint,rain_mm,wind_speed,wind_direction,pressure,weather_code
665606,2025-12-06 14:00:00+07:00,26.338000,85.643456,23.737999,0.9,3.438895,312.878900,1006.91560,53.0
665607,2025-12-06 15:00:00+07:00,26.737999,85.940410,24.188000,0.3,0.540000,180.000000,1006.32007,51.0
665608,2025-12-06 16:00:00+07:00,26.938000,84.427315,24.088000,0.2,3.244996,146.309900,1006.42114,51.0
665609,2025-12-06 17:00:00+07:00,26.138000,83.076805,23.038000,0.2,5.081613,157.067870,1006.21594,51.0
665610,2025-12-06 18:00:00+07:00,25.538000,89.797640,23.737999,0.2,2.545584,81.869990,1006.61066,51.0
665611,2025-12-06 19:00:00+07:00,25.538000,92.813650,24.288000,0.0,2.346913,32.471172,1007.40890,3.0
665612,2025-12-06 20:00:00+07:00,25.288000,94.769290,24.388000,0.1,2.811690,50.194473,1008.20526,51.0
665613,2025-12-06 21:00:00+07:00,25.138000,95.906000,24.438000,0.0,3.864764,27.758451,1008.70325,3.0
665614,2025-12-06 22:00:00+07:00,24.688000,95.031770,23.838000,0.2,0.917824,11.309895,1008.89930,51.0
665615,2025-12-06 23:00:00+07:00,24.688000,93.330620,23.538000,0.3,1.451206,209.744800,1008.79950,51.0


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 665616 entries, 0 to 665615
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   date            665616 non-null  object 
 1   temperature     665616 non-null  float64
 2   humidity        665616 non-null  float64
 3   dewpoint        665616 non-null  float64
 4   rain_mm         665616 non-null  float64
 5   wind_speed      665616 non-null  float64
 6   wind_direction  665616 non-null  float64
 7   pressure        665616 non-null  float64
 8   weather_code    665616 non-null  float64
dtypes: float64(8), object(1)
memory usage: 45.7+ MB
